# Minimax i Alfa-Beta dla gry Clobber

---

## 1. Wprowadzenie teoretyczne

### 1.1 Drzewo decyzyjne
Drzewo decyzyjne reprezentowane jest przez stany i połączenia pomiędzy nimi. Każdy węzeł to stan gry, krawędzie to możliwe ruchy.

### 1.2 Algorytm Minimax
Minimax to rekurencyjna metoda przeglądu drzewa, wybierająca dla graczy wartości maksymalną lub minimalną.

### 1.3 Cięcie alfa-beta
Cięcie alfa-beta optymalizuje Minimax, pomijając gałęzie, które nie wpłyną na wynik.

---

## 2. Implementacja podstawowa

# Importy i konfiguracja globalna

In [1]:
import time
import copy
from typing import List, Tuple, Optional

# Globalny licznik odwiedzonych węzłów
node_counter = 0

### 2.1 Definicja struktury drzewa (opcjonalna)

In [2]:
class DecisionNode:
    """
    Reprezentuje węzeł w drzewie decyzyjnym.
    """
    def __init__(self, state, children=None, player=None):
        self.state = state
        self.children = children if children is not None else []
        self.player = player 
        self.value = None

### 2.2 Implementacja Minimax

In [3]:
def minimax(node: DecisionNode, depth: int, is_maximizing: bool) -> float:
    global node_counter
    node_counter += 1

    if depth == 0 or not node.children:
        return node.value  

    if is_maximizing:
        best = float('-inf')
        for child in node.children:
            val = minimax(child, depth-1, False)
            best = max(best, val)
        return best
    else:
        best = float('inf')
        for child in node.children:
            val = minimax(child, depth-1, True)
            best = min(best, val)
        return best

### 2.3 Implementacja Minimax z alfa-beta

In [4]:
def alphabeta(node: DecisionNode, depth: int, alpha: float, beta: float, is_maximizing: bool) -> float:
    global node_counter
    node_counter += 1
    if depth == 0 or not node.children:
        return node.value

    if is_maximizing:
        value = float('-inf')
        for child in node.children:
            value = max(value, alphabeta(child, depth-1, alpha, beta, False))
            alpha = max(alpha, value)
            if alpha >= beta:
                break  # beta cut-off
        return value
    else:
        value = float('inf')
        for child in node.children:
            value = min(value, alphabeta(child, depth-1, alpha, beta, True))
            beta = min(beta, value)
            if beta <= alpha:
                break  # alpha cut-off
        return value


## 3. Zadania: Gra Clobber

In [5]:
from typing import List
from typing import Tuple

class ClobberState:
    def __init__(self, board: List[List[str]], current_player: str = 'B'):
        self.board = board
        self.current_player = current_player  # 'B' lub 'W'
        self.n = len(board[0])
        self.m = len(board)

    def get_opponent(self) -> str:
        return 'W' if self.current_player == 'B' else 'B'

    def in_bounds(self, r: int, c: int) -> bool:
        return 0 <= r < self.m and 0 <= c < self.n

    def generate_moves(self) -> List[Tuple[int,int,int,int]]:
        """Zwraca listę ruchów jako krotek (r_from, c_from, r_to, c_to)."""
        moves = []
        dirs = [(-1,0),(1,0),(0,-1),(0,1)]
        for r in range(self.m):
            for c in range(self.n):
                if self.board[r][c] == self.current_player:
                    for dr,dc in dirs:
                        nr, nc = r+dr, c+dc
                        if self.in_bounds(nr,nc) and self.board[nr][nc] == self.get_opponent():
                            moves.append((r,c,nr,nc))
        return moves

    def apply_move(self, move: Tuple[int,int,int,int]) -> 'ClobberState':
        r,c,nr,nc = move
        new_board = copy.deepcopy(self.board)
        new_board[nr][nc] = self.current_player
        new_board[r][c] = '_'
        return ClobberState(new_board, current_player=self.get_opponent())

    def is_terminal(self) -> bool:
        return len(self.generate_moves()) == 0

    def __str__(self):
        return '\n'.join(' '.join(row) for row in self.board)

### 3.2 Heurystyki oceny stanu gry

Zaimplementujemy trzy heurystyki:
1. **Różnica pionków**: liczba własnych minus liczba przeciwnika.
2. **Liczba dostępnych ruchów**: liczba ruchów własnych minus przeciwnika.
3. **Waga pozycyjna**: pilnujemy krawędzi.

In [6]:
def heuristic_piece_diff(state: ClobberState) -> int:
    b = sum(row.count('B') for row in state.board)
    w = sum(row.count('W') for row in state.board)
    return b - w if state.current_player=='B' else w - b

def heuristic_mobility(state: ClobberState) -> int:
    own_moves = len(state.generate_moves())
    # zmień gracza, zbadaj ruchy przeciwnika
    opp_state = ClobberState(state.board, state.get_opponent())
    opp_moves = len(opp_state.generate_moves())
    return own_moves - opp_moves

def heuristic_positional(state: ClobberState) -> int:
    weight = 0
    m,n = state.m, state.n
    for r in range(m):
        for c in range(n):
            if state.board[r][c] == state.current_player:
                # krawędź zwiększa wagę
                if r in [0,m-1] or c in [0,n-1]:
                    weight += 2
                else:
                    weight += 1
            elif state.board[r][c] == state.get_opponent():
                if r in [0,m-1] or c in [0,n-1]:
                    weight -= 2
                else:
                    weight -= 1
    return weight

### 3.3 Agent Minimax i alfa-beta dla Clobber

In [7]:
def evaluate(state: ClobberState, heuristic) -> float:
    return heuristic(state)

def minimax_clobber(state: ClobberState, depth: int, heuristic):
    """Zwraca najlepszy ruch i jego wartość."""
    global node_counter
    node_counter = 0
    start = time.time()

    def recurse(s: ClobberState, d: int, maximizing: bool) -> float:
        global node_counter
        node_counter += 1
        if d == 0 or s.is_terminal():
            return evaluate(s, heuristic)
        moves = s.generate_moves()
        if not moves:
            return evaluate(s, heuristic)
        if maximizing:
            best_val = float('-inf')
            for mv in moves:
                val = recurse(s.apply_move(mv), d-1, False)
                best_val = max(best_val, val)
            return best_val
        else:
            best_val = float('inf')
            for mv in moves:
                val = recurse(s.apply_move(mv), d-1, True)
                best_val = min(best_val, val)
            return best_val

    best_move = None
    best_value = float('-inf')
    for mv in state.generate_moves():
        val = recurse(state.apply_move(mv), depth-1, False)
        if val > best_value:
            best_value = val
            best_move = mv
    end = time.time()
    print(f"Minimax: ruch={best_move}, wartość={best_value}, odwiedzone węzły={node_counter}, czas={end-start:.4f}s")
    return best_move, best_value

def alphabeta_clobber(state: ClobberState, depth: int, heuristic):
    global node_counter
    node_counter = 0
    start = time.time()

    def recurse(s: ClobberState, d: int, alpha: float, beta: float, maximizing: bool) -> float:
        global node_counter
        node_counter += 1
        if d == 0 or s.is_terminal():
            return evaluate(s, heuristic)
        moves = s.generate_moves()
        if not moves:
            return evaluate(s, heuristic)
        if maximizing:
            value = float('-inf')
            for mv in moves:
                value = max(value, recurse(s.apply_move(mv), d-1, alpha, beta, False))
                alpha = max(alpha, value)
                if alpha >= beta:
                    break
            return value
        else:
            value = float('inf')
            for mv in moves:
                value = min(value, recurse(s.apply_move(mv), d-1, alpha, beta, True))
                beta = min(beta, value)
                if beta <= alpha:
                    break
            return value

    best_move = None
    best_value = float('-inf')
    alpha, beta = float('-inf'), float('inf')
    for mv in state.generate_moves():
        val = recurse(state.apply_move(mv), depth-1, alpha, beta, False)
        if val > best_value:
            best_value = val
            best_move = mv
        alpha = max(alpha, best_value)
    end = time.time()
    print(f"Alpha-Beta: ruch={best_move}, wartość={best_value}, odwiedzone węzły={node_counter}, czas={end-start:.4f}s")
    return best_move, best_value

## 4. Przykład rozgrywki (wersja podstawowa)

In [8]:
def default_board(m=5, n=6) -> List[List[str]]:
    board = []
    for r in range(m):
        row = []
        for c in range(n):
            if (r + c) % 2 == 0:
                row.append('B')
            else:
                row.append('W')
        board.append(row)
    return board

state = ClobberState(default_board())
print(state)
print("Startuje gracz:", state.current_player)

B W B W B W
W B W B W B
B W B W B W
W B W B W B
B W B W B W
Startuje gracz: B


# Teraz agent gra samodzielnie do końca gry

In [9]:
def play_full_game(depth: int, heuristic):
    s = ClobberState(default_board())
    rounds = 0
    while True:
        if s.is_terminal():
            break
        mv, _ = alphabeta_clobber(s, depth, heuristic)
        if mv is None:
            break
        s = s.apply_move(mv)
        rounds += 1
    print("Koniec gry po rundach:", rounds)
    print(s)
    print("Zwycięzca:", s.get_opponent())


# Uruchomienie przykładu:

In [10]:
play_full_game(depth=3, heuristic=heuristic_piece_diff)

Alpha-Beta: ruch=(0, 0, 1, 0), wartość=-1, odwiedzone węzły=2039, czas=0.0872s
Alpha-Beta: ruch=(0, 1, 1, 1), wartość=0, odwiedzone węzły=1704, czas=0.0893s
Alpha-Beta: ruch=(0, 2, 1, 2), wartość=-1, odwiedzone węzły=1485, czas=0.0836s
Alpha-Beta: ruch=(0, 3, 1, 3), wartość=0, odwiedzone węzły=1281, czas=0.0482s
Alpha-Beta: ruch=(0, 4, 1, 4), wartość=-1, odwiedzone węzły=1140, czas=0.0739s
Alpha-Beta: ruch=(0, 5, 1, 5), wartość=0, odwiedzone węzły=1040, czas=0.0555s
Alpha-Beta: ruch=(1, 0, 1, 1), wartość=-1, odwiedzone węzły=974, czas=0.0468s
Alpha-Beta: ruch=(1, 3, 1, 2), wartość=0, odwiedzone węzły=836, czas=0.0261s
Alpha-Beta: ruch=(1, 1, 2, 1), wartość=-1, odwiedzone węzły=796, czas=0.0232s
Alpha-Beta: ruch=(1, 2, 2, 2), wartość=0, odwiedzone węzły=653, czas=0.0181s
Alpha-Beta: ruch=(1, 4, 1, 5), wartość=-1, odwiedzone węzły=567, czas=0.0160s
Alpha-Beta: ruch=(2, 2, 2, 1), wartość=0, odwiedzone węzły=500, czas=0.0148s
Alpha-Beta: ruch=(1, 5, 2, 5), wartość=-1, odwiedzone węzły=491,

## 5. Wersja rozszerzona

Wersja umożliwia partię między dwoma agentami z różnymi heurystykami.

In [11]:
def play_two_agents(depth: int, heuristic1, heuristic2):
    s = ClobberState(default_board())
    rounds = 0
    while True:
        if s.is_terminal():
            break
        if s.current_player == 'B':
            mv, _ = alphabeta_clobber(s, depth, heuristic1)
        else:
            mv, _ = alphabeta_clobber(s, depth, heuristic2)
        if mv is None:
            break
        s = s.apply_move(mv)
        rounds += 1
    print("Koniec gry po rundach:", rounds)
    print(s)
    print("Zwycięzca:", s.get_opponent())

In [12]:
play_two_agents(depth=3, heuristic1=heuristic_piece_diff, heuristic2=heuristic_mobility)

Alpha-Beta: ruch=(0, 0, 1, 0), wartość=-1, odwiedzone węzły=2039, czas=0.0578s
Alpha-Beta: ruch=(0, 1, 1, 1), wartość=0, odwiedzone węzły=1704, czas=0.1154s
Alpha-Beta: ruch=(0, 2, 1, 2), wartość=-1, odwiedzone węzły=1485, czas=0.0493s
Alpha-Beta: ruch=(0, 3, 1, 3), wartość=0, odwiedzone węzły=1281, czas=0.1132s
Alpha-Beta: ruch=(0, 4, 1, 4), wartość=-1, odwiedzone węzły=1140, czas=0.0308s
Alpha-Beta: ruch=(0, 5, 1, 5), wartość=0, odwiedzone węzły=1040, czas=0.0631s
Alpha-Beta: ruch=(1, 0, 1, 1), wartość=-1, odwiedzone węzły=974, czas=0.0259s
Alpha-Beta: ruch=(1, 3, 1, 2), wartość=0, odwiedzone węzły=836, czas=0.0461s
Alpha-Beta: ruch=(1, 1, 2, 1), wartość=-1, odwiedzone węzły=796, czas=0.0202s
Alpha-Beta: ruch=(1, 2, 2, 2), wartość=0, odwiedzone węzły=653, czas=0.0348s
Alpha-Beta: ruch=(1, 4, 1, 5), wartość=-1, odwiedzone węzły=567, czas=0.0150s
Alpha-Beta: ruch=(2, 2, 2, 1), wartość=0, odwiedzone węzły=500, czas=0.0256s
Alpha-Beta: ruch=(1, 5, 2, 5), wartość=-1, odwiedzone węzły=491,